In [ ]:
import os
import json
import glob
import re
import datetime
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
import tkinter as tk
from tkinter import ttk, filedialog, messagebox

pio.renderers.default = "notebook"


In [ ]:
class KingKevEISAnalyzer:
    def __init__(self, folders, config):
        self.folders = folders # Now accepts a list
        self.config = config
        self.log_paths = []
        self.test_name = "Unknown_Test"
        self.testbench = "70804 / KingKev"
        
        # Multi-Channel Storage Structure
        self.channels = {i: {'id': f'Unknown_CH{i}', 'area': 1.0, 'true_ocv': None, 'df': pd.DataFrame()} for i in range(1, 5)}
        self.timeline = []
        self.figures = {}
        self.colors = self.config.get("custom_colors", px.colors.qualitative.Plotly) * 3

    def _find_true_ocv(self, ch_num, log_path):
        """Extracts the first True OCV measurement for a specific channel from a given log file."""
        with open(log_path, 'r', encoding='latin1') as f:
            lines = f.readlines()
            
        found_ocv = None
        try:
            start_str = f"Sub Eis_Ocv_Channel_{ch_num}, START"
            start_index = next(i for i, line in enumerate(lines) if start_str in line)
            
            # KingKev logs voltage as: "voltage CH01: 1,009...V"
            voltage_pattern = re.compile(rf'voltage CH0{ch_num}:\s*([\d,.]+)')
            for i in range(start_index, len(lines)):
                match = voltage_pattern.search(lines[i])
                if match:
                    found_ocv = float(match.group(1).replace(',', '.'))
                    break
        except StopIteration:
            pass
            
        if found_ocv is not None:
            self.channels[ch_num]['true_ocv'] = found_ocv
            print(f"  -> Found definitive OCV for CH {ch_num}: {found_ocv:.5f} V")
            return True
        return False

    def parse_master_log(self):
        print("\n--- 1. Parsing KingKev Master Log(s) ---")
        metadata_extracted = False
        
        for folder in self.folders:
            log_files = glob.glob(os.path.join(folder, "*LogFile*.txt")) + glob.glob(os.path.join(folder, "*.log"))
            if not log_files: 
                print(f"  -> WARNING: No log file found in folder: {folder}")
                continue
                
            log_path = log_files[0]
            self.log_paths.append(log_path)
            print(f"  -> Found log file: {os.path.basename(log_path)}")
            
            with open(log_path, 'r', encoding='latin1') as f: 
                content = f.read()
            
            # 1. Extract Test Name, Channels, & Area (Only need to do this once from the first valid file)
            if not metadata_extracted:
                test_name_match = re.search(r'Test-Name:\s*(.*)', content, re.IGNORECASE)
                tb_match = re.search(r'Test bench:\s*(.*)', content, re.IGNORECASE)
                if test_name_match: self.test_name = test_name_match.group(1).strip()
                if tb_match: self.testbench = tb_match.group(1).strip()
                    
                for i in range(1, 5):
                    pattern = rf'Cell-Name-Channel {i}:\s*([A-Za-z0-9_-]+),\s*Area:\s*([\d,.]+)'
                    match = re.search(pattern, content, re.IGNORECASE)
                    if match:
                        self.channels[i]['id'] = match.group(1).strip()
                        self.channels[i]['area'] = float(match.group(2).replace(',', '.'))
                        print(f"  -> Extracted Channel {i}: {self.channels[i]['id']} (Area: {self.channels[i]['area']} cm²)")
                metadata_extracted = True
            
            # 2. Extract True OCVs (Check until we find one for each channel)
            for i in range(1, 5):
                if self.channels[i]['true_ocv'] is None:
                    self._find_true_ocv(i, log_path)

            # 3. Build Timeline (Line by Line to maintain State)
            current_temp, current_hum, current_flow = 650, 0, "High Flow" # Default hum to 0 for initial start up
            
            with open(log_path, 'r', encoding='latin1') as f:
                for line in f:
                    flow_match = re.search(r'(?:Set H2 flow to|Anode hydrogen \((?:H2)\)):\s*([\d.,]+)', line, re.IGNORECASE)
                    if flow_match:
                        val = float(flow_match.group(1).replace(',', '.'))
                        flow_ml = val if val > 50 else val * 1000
                        current_flow = "High Flow" if flow_ml > 300 else "Low Flow"
                        
                    if "Set O2 flow to 0 " in line or "O2: 0" in line:
                        current_hum = 0
                        
                    env_match = re.search(r'(\d{2}\.\d{2}\.\d{4}\s\d{2}:\d{2}:\d{2})pH2O\s*=\s*(\d+)%,\s*T=([\d.,]+)', line)
                    if env_match:
                        dt_str, hum_str, temp_str = env_match.groups()
                        dt = datetime.datetime.strptime(dt_str, "%d.%m.%Y %H:%M:%S")
                        current_hum = int(hum_str)
                        raw_temp = float(temp_str.replace(',', '.'))
                        current_temp = int(round(raw_temp / 25.0) * 25.0)
                        self.timeline.append({'timestamp': dt, 'type': 'env', 'temperature': current_temp, 'humidity': current_hum, 'flow_regime': current_flow})
                        
                    eis_start_pattern = re.search(r'[$]?(\d{2}\.\d{2}\.\d{4}\s\d{2}:\d{2}:\d{2}), Sub Eis_Ocv_Channel_(\d+), START', line)
                    if eis_start_pattern:
                        dt = datetime.datetime.strptime(eis_start_pattern.group(1), "%d.%m.%Y %H:%M:%S")
                        ch = int(eis_start_pattern.group(2))
                        self.timeline.append({'timestamp': dt, 'type': 'eis_start', 'ch': ch, 'temperature': current_temp, 'humidity': current_hum, 'flow_regime': current_flow})
                        
        # Sort unified timeline to stitch parts together seamlessly
        self.timeline.sort(key=lambda x: x['timestamp'])
        print(f"  -> Extracted {len(self.timeline)} total timeline events across all folders.")


    def read_impedance_file(self, file_path, area):
        real_part, imag_part, frequencies = [], [], []
        try:
            with open(file_path, 'r', encoding='latin1') as file: lines = file.readlines()
            data_start = next((i + 4 for i, l in enumerate(lines) if 'Impedance samples' in l), -1)
            if data_start == -1: return None, None, None
            
            for j in range(data_start, len(lines)):
                if lines[j].strip():
                    parts = lines[j].split()
                    if len(parts) >= 4:
                        try:
                            freq, r_val, i_val = float(parts[1]), float(parts[2]), float(parts[3])
                            if r_val >= 0:
                                frequencies.append(freq)
                                real_part.append(r_val * area)
                                imag_part.append(-i_val * area)   
                        except (ValueError, IndexError): continue
            return np.array(real_part), np.array(imag_part), np.array(frequencies)
        except Exception: 
            return None, None, None

    def _get_crossings(self, raw_r, raw_i):
        crossings = []
        for idx in range(len(raw_i) - 1):
            y1, y2 = raw_i[idx], raw_i[idx+1]
            x1, x2 = raw_r[idx], raw_r[idx+1]
            
            # Check if the line segment between these two dots crosses y=0 
            if (y1 <= 0 and y2 > 0) or (y1 > 0 and y2 <= 0):
                if y2 != y1:
                    x_cross = x1 + (0 - y1) * (x2 - x1) / (y2 - y1)
                    crossings.append(x_cross)
        
        if len(crossings) >= 2:
            exact_rs = min(crossings)
            exact_rtot = max(crossings)
        elif len(crossings) == 1:
            exact_rs = crossings[0]
            pos_indices = [x for x, y in zip(raw_r, raw_i) if y >= 0]
            exact_rtot = pos_indices[-1] if pos_indices else exact_rs
        else:
            pos_indices = [x for x, y in zip(raw_r, raw_i) if y >= 0]
            exact_rs = pos_indices[0] if pos_indices else None
            exact_rtot = pos_indices[-1] if pos_indices else None
            
        return exact_rs, exact_rtot

    def parse_eis_files(self):
        print("\n--- 2. Parsing EIS Data & Mapping to Unified Timeline ---")
        
        # Gather EIS files from ALL folders
        all_eis_files = []
        for folder in self.folders:
            eis_dir = os.path.join(folder, 'Data', 'EIS_orig')
            if not os.path.exists(eis_dir):
                eis_dir = os.path.join(folder, 'EIS_orig')
                
            if os.path.exists(eis_dir):
                all_eis_files.extend(glob.glob(os.path.join(eis_dir, "*.txt")))
            else:
                print(f"  -> WARNING: No EIS_orig folder found in {folder}.")
                
        if not all_eis_files:
            print("  -> ERROR: No EIS files found in any selected folders.")
            return

        raw_data = {1: [], 2: [], 3: [], 4: []}
        
        for file_path in all_eis_files:
            filename = os.path.basename(file_path)
            match = re.search(r'(\d{12})', filename, re.IGNORECASE)
            if not match: continue
            
            ts_str = match.group(1)
            try:
                year, month, day = int("20" + ts_str[0:2]), int(ts_str[2:4]), int(ts_str[4:6])
                hour, minute, second = int(ts_str[6:8]), int(ts_str[8:10]), int(ts_str[10:12])
                file_time = datetime.datetime(year, month, day, hour, minute, second)
            except Exception: continue
            
            current_temp, current_hum, current_flow, current_ch = 650, 0, "High Flow", 1
            
            for event in self.timeline:
                if event['timestamp'] <= file_time:
                    if event['type'] == 'env':
                        current_temp = event['temperature']
                        current_hum = event['humidity']
                        current_flow = event.get('flow_regime', "High Flow")
                    elif event['type'] == 'eis_start':
                        current_ch = event['ch']
                else: 
                    break
                    
            ch_match = re.search(r'CH(\d+)', filename, re.IGNORECASE)
            if ch_match:
                current_ch = int(ch_match.group(1))
            
            if current_ch not in raw_data: continue
            
            r_val, i_val, freqs = self.read_impedance_file(file_path, self.channels[current_ch]['area'])
            if r_val is not None and len(r_val) > 0:
                raw_data[current_ch].append({
                    'filename': filename, 
                    'timestamp': file_time, 
                    'temperature': current_temp, 
                    'humidity': current_hum, 
                    'flow_regime': current_flow, 
                    'Re(Z)': r_val, 
                    '-Im(Z)': i_val, 
                    'Frequency': freqs
                })
                
        total_valid_curves = 0
        for ch_num in range(1, 5):
            if raw_data[ch_num]:
                self.channels[ch_num]['df'] = pd.DataFrame(raw_data[ch_num]).sort_values('timestamp')
                print(f"  -> Channel {ch_num} ({self.channels[ch_num]['id']}): {len(raw_data[ch_num])} valid EIS curves loaded.")
                total_valid_curves += len(raw_data[ch_num])
            
        if total_valid_curves == 0:
            print("\n  !!! WARNING: Zero valid EIS curves loaded. Check your EIS_orig folder! !!!")


    def _generate_eis_table(self, title, plot_df, cell_id, var_type):
        headers = [
            [f"<b>{var_type.capitalize()}</b>", ""], 
            ["<b>Flow Regime</b>", "<i> High / Low </i>"], 
            ["<b>R<sub>s</sub></b>", "<i>(Ω·cm²)</i>"], 
            ["<b>R<sub>pol</sub></b>", "<i>(Ω·cm²)</i>"]
        ]
        
        row_data = [[], [], [], []]
        
        for _, row in plot_df.iterrows():
            rs, rtot = self._get_crossings(row['Re(Z)'], row['-Im(Z)'])
            rpol = (rtot - rs) if rs is not None and rtot is not None else None
            
            if var_type == 'humidity':
                row_data[0].append(f"{row['humidity']}%")
            else:
                row_data[0].append(f"{row['temperature']}°C")
                
            row_data[1].append(row['flow_regime'])
            row_data[2].append(f"{rs:.4f}" if rs is not None else "-")
            row_data[3].append(f"{rpol:.4f}" if rpol is not None else "-")
            
        fig = go.Figure(data=[go.Table(
            columnwidth=[0.8, 1.2, 1.0, 1.0], 
            header=dict(
                values=headers, 
                fill_color='#E1F0FA', # Soft baby blue
                font=dict(color='#212F3D', size=15, family="Arial"), 
                align='center',
                line_color='#D5D8DC', 
                line_width=1, 
                height=35 # Taller header space
            ), 
            cells=dict(
                values=row_data, 
                fill_color='white', 
                font=dict(color='#2C3E50', size=15, family="Arial"), 
                align='center', 
                line_color='#D5D8DC', 
                line_width=1,
                height=35 # Taller cell space
            )
        )])
        
        # FIX: Safer height calculation to prevent vertical squashing & scrollbars
        # 200px (Title + Margins + Headers) + 35px per row of data
        calculated_height = 200 + (len(row_data[0]) * 35)
        
        fig.update_layout(
            title=dict(text=title, font=dict(family="Arial", size=18, color="black")), 
            height=calculated_height, 
            width=1000, # Increased width for better proportions
            margin=dict(t=60, b=20, l=20, r=20), # Added side margins
            paper_bgcolor='rgba(0,0,0,0)'
        )
        self.figures[title] = fig



    def generate_kingkev_plots(self, jc_mode=False):
        print("\n--- 3. Generating KingKev Analyst Footprint Sequences ---")
        if jc_mode:
            print("  -> JC Mode Active: Keeping 1:1 aspect ratio but cropping figure vertically...")
            
        for ch_num, ch_data in self.channels.items():
            df = ch_data['df']
            cell_id = ch_data['id']
            if df.empty: continue
            
            # Sub-Select Specific Curves
            h2o_0 = df[(df['humidity'] == 0) & (df['temperature'] == 650)].head(1)
            h2o_30 = df[(df['humidity'] == 30) & (df['temperature'] == 650)].head(1)
            h2o_50_all = df[(df['humidity'] == 50) & (df['temperature'] == 650) & (df['flow_regime'] == 'High Flow')]
            h2o_50_initial = h2o_50_all.head(1)
            if h2o_50_initial.empty: h2o_50_initial = df[(df['humidity'] == 50) & (df['temperature'] == 650)].head(1)
            
            h2o_70 = df[(df['humidity'] == 70) & (df['temperature'] == 650)].head(1)
            h2o_90_high = df[(df['humidity'] == 90) & (df['temperature'] == 650) & (df['flow_regime'] == 'High Flow')].head(1)
            h2o_90_low = df[(df['humidity'] == 90) & (df['temperature'] == 650) & (df['flow_regime'] == 'Low Flow')].head(1)
            
            h2o_50_post_hum = pd.DataFrame()
            if not h2o_90_low.empty:
                h2o_50_post_hum = h2o_50_all[h2o_50_all['timestamp'] > h2o_90_low.iloc[0]['timestamp']].head(1)
            elif not h2o_90_high.empty:
                h2o_50_post_hum = h2o_50_all[h2o_50_all['timestamp'] > h2o_90_high.iloc[0]['timestamp']].head(1)
                
            temp_var_df = df[df['humidity'] == 50].copy()
            if not temp_var_df.empty:
                temp_var_df = temp_var_df[temp_var_df['temperature'].isin([750, 725, 700, 675, 650, 625, 600])]
                temp_var_df = temp_var_df.drop_duplicates(subset=['temperature'], keep='last').sort_values('temperature', ascending=False)
                
            h2o_50_final = df[(df['temperature'] == 650) & (df['humidity'] == 50)].tail(1)
            
            true_ocv = self.channels[ch_num].get('true_ocv', None)
            ocv_str = f"{true_ocv:.3f} V" if true_ocv else "Unknown OCV"
            
            # 1. 0% H2O
            if not h2o_0.empty:
                title = f"EIS for {cell_id} (CH{ch_num}) at 650°C (100% H₂) with an OCV of {ocv_str}"
                fig = self._create_base_figure(title)
                self._add_trace(fig, h2o_0.iloc[0], f"100% H₂ / 0% H₂O", 0)
                self.figures[title] = fig

            # 2. 0, 30, 50 Variation
            plot2_df = pd.concat([h2o_0, h2o_30, h2o_50_initial])
            if len(plot2_df) > 1:
                title = f"EIS Humidity Variation for {cell_id} (CH{ch_num}) at 650°C (0%, 30% & 50%)"
                fig = self._create_base_figure(title)
                for i, row in plot2_df.reset_index().iterrows():
                    self._add_trace(fig, row, f"{row['humidity']}% H₂O ({row['flow_regime']})", i)
                self.figures[title] = fig

            # 3. 30, 50, 70, 90 Variation
            plot3_df = pd.concat([h2o_30, h2o_50_initial, h2o_70, h2o_90_high, h2o_90_low])
            if len(plot3_df) > 1:
                title = f"EIS Humidity Variation for {cell_id} (CH{ch_num}) at 650°C"
                fig = self._create_base_figure(title) 
                for i, row in plot3_df.reset_index().iterrows():
                    self._add_trace(fig, row, f"{row['humidity']}% H₂O ({row['flow_regime']})", i)
                self.figures[title] = fig
                self._generate_eis_table(f"Resistances Table: Humidity Variation {cell_id} (CH{ch_num})", plot3_df, cell_id, "humidity")


            # 4. Stability Check
            plot4_df = pd.concat([h2o_50_initial, h2o_50_post_hum])
            if len(plot4_df) == 2:
                title = f"EIS Stability Check for {cell_id} (CH{ch_num}) after High Humidity"
                fig = self._create_base_figure(title)
                self._add_trace(fig, plot4_df.iloc[0], "Baseline 50% H₂O", 0)
                self._add_trace(fig, plot4_df.iloc[1], "Post-Hum 50% H₂O", 1)
                self.figures[title] = fig

            # 5. Temperature Variation
            if len(temp_var_df) > 1:
                title = f"EIS Temperature Variation for {cell_id} (CH{ch_num})"
                fig = self._create_base_figure(title)
                for i, row in temp_var_df.reset_index().iterrows():
                    self._add_trace(fig, row, f"{row['temperature']}°C - {row['humidity']}% H₂O ({row['flow_regime']})", i)
                self.figures[title] = fig
                self._generate_eis_table(f"Resistances Table: Temp Variation {cell_id} (CH{ch_num})", temp_var_df, cell_id, "temperature")

            # 6. Aging Check
            plot6_df = pd.concat([h2o_50_initial, h2o_50_final])
            if len(plot6_df) == 2 and (h2o_50_initial.iloc[0]['timestamp'] != h2o_50_final.iloc[0]['timestamp']):
                title = f"EIS Aging of {cell_id} (CH{ch_num}) at 50% H₂O and 650°C"
                fig = self._create_base_figure(title)
                self._add_trace(fig, plot6_df.iloc[0], f"Initial Reference (BoT)", 0)
                self._add_trace(fig, plot6_df.iloc[1], f"Final Check (EoT)", 1)
                self.figures[title] = fig
                
        print(f"\n--- Outputting {len(self.figures)} Objects to Browser ---")
        
        for title, fig in self.figures.items():
            is_table = "Table" in title
            
            # Skip optimizing axes for tables so it doesn't crash
            if not is_table:
                self._optimize_axes(fig, jc_mode=jc_mode)
                
            safe_filename = title.replace(')', '').replace('(', '').replace('°C', 'C').replace('/', '_').replace(' ', '_')
            
            if is_table:
                # Always force SVG for Tables for perfect text quality
                download_config = {
                    'toImageButtonOptions': {
                        'format': 'svg',          
                        'filename': safe_filename,
                        'height': fig.layout.height, 
                        'width': fig.layout.width,
                        'scale': 1 # SVG is vector, so scale 1 is infinitely sharp
                    }
                }
            elif jc_mode:
                # JC Mode: Download as high-res PNG keeping the dynamically cropped layout size
                current_width = fig.layout.width
                current_height = fig.layout.height
                
                download_config = {
                    'toImageButtonOptions': {
                        'format': 'png',          
                        'filename': safe_filename,
                        'height': current_height, 
                        'width': current_width,   
                        'scale': 3                
                    }
                }
            else:
                # Normal Mode: Download as scalable SVG using uncropped base size
                download_config = {
                    'toImageButtonOptions': {
                        'format': 'svg',          
                        'filename': safe_filename,
                        'height': fig.layout.height or 900, 
                        'width': fig.layout.width or 1400,
                        'scale': 1 
                    }
                }
                
            fig.show(config=download_config)


    def _create_base_figure(self, title):
        fig = go.Figure()
        fig.update_layout(
            title=dict(text=title, font=dict(family="Arial", size=26, color="black")),
            template="plotly_white", 
            height=900, width=1400, 
            hovermode="closest", 
            margin=dict(l=80, r=80, t=100, b=80), # Margin tightened on right
            legend=dict(
                x=0.98, y=0.98, xanchor="right", yanchor="top", # Legend moved inside top-right
                bordercolor="#888", borderwidth=1, 
                bgcolor="rgba(255,255,255,0.9)", font=dict(family="Arial", size=16, color="#333"), 
                itemclick="toggle", itemdoubleclick="toggleothers"
            ),
            xaxis=dict(
                title=dict(text="Re(Z) / Ω·cm<sup>2</sup>", font=dict(size=20)), 
                tickfont=dict(size=18), showgrid=True, gridcolor="#E5E5E5", zerolinecolor="#888"
            ),
            yaxis=dict(
                title=dict(text="-Im(Z) / Ω·cm<sup>2</sup>", font=dict(size=20)), 
                tickfont=dict(size=18), showgrid=True, gridcolor="#E5E5E5", zerolinecolor="#888", 
                scaleanchor="x", scaleratio=1
            ),
        )
        fig.add_hline(y=0.0, line_width=1, line_color="black")
        return fig

    def _add_trace(self, fig, row, name, color_idx):
        color = self.colors[color_idx % len(self.colors)]
        
        re_arr, im_arr = row['Re(Z)'], row['-Im(Z)']
        
        fig.add_trace(go.Scatter(
            x=re_arr, y=im_arr, 
            name=name, mode='markers+lines', 
            marker=dict(size=8, color=color, symbol='circle'), 
            line=dict(width=2.5, color=color),
            hovertemplate=f"<b>{name}</b><br>Re(Z): %{{x:.4f}} Ω·cm²<br>-Im(Z): %{{y:.4f}} Ω·cm²<br>Freq: %{{text:.1f}} Hz<extra></extra>", 
            text=row['Frequency']
        ))
        
    def _optimize_axes(self, fig, jc_mode=False):
        all_x, all_y = [], []
        for trace in fig.data:
            if trace.x is not None:
                all_x.extend(trace.x)
            if trace.y is not None:
                all_y.extend(trace.y)
                
        if all_x and all_y:
            min_x, max_x = min(all_x), max(all_x)
            min_y, max_y = min(all_y), max(all_y)
            
            x_range = max_x - min_x
            y_range = max_y - min_y
            
            if x_range == 0: x_range = 1.0
            if y_range == 0: y_range = 1.0
            
            pad_x = x_range * 0.1
            pad_y = y_range * 0.1
            
            y_lower_bound = min_y - pad_y
            
            if jc_mode:
                # Account for inductive tails dropping below 0
                y_lower_bound = min(0, min_y - pad_y) 
                
                # Calculate true mathematical ranges for the exact bounding box
                actual_x_range = (max_x + pad_x) - (min_x - pad_x)
                actual_y_range = (max_y + pad_y) - y_lower_bound
                
                plot_width = 1400 - 80 - 80 
                
                # Calculate the EXACT proportional height for the plot area (Rectangular Crop)
                target_plot_height = plot_width * (actual_y_range / actual_x_range)
                
                # Set the EXACT figure height to maintain the strict rectangle
                # Using a safe minimum of 400px so it never squashes the UI
                target_figure_height = int(target_plot_height + 180)
                fig.update_layout(height=max(400, target_figure_height))
                
            fig.update_xaxes(range=[min_x - pad_x, max_x + pad_x])
            fig.update_yaxes(range=[y_lower_bound, max_y + pad_y])


In [ ]:
SESSION_FILE = "kingkev_session.json"

def main():
    try:
        from ctypes import windll
        windll.shcore.SetProcessDpiAwareness(1)
    except Exception: 
        pass

    try:
        import customtkinter as ctk
    except ImportError:
        print("\n[!] ERROR: CustomTkinter is not installed.")
        print("Please run: pip install customtkinter")
        return

    print("\n==============================================")
    print("  KingKev (4-Channel) EIS Data Analyzer")
    print("==============================================\n")

    # ==========================================
    # PURE MONOCHROME + BABY BLUE
    # ==========================================
    bg_color = ("#FAFAFA", "#09090B")         # Off-white / Matte Black
    card_color = ("#FFFFFF", "#121214")       # Pure White / Very Dark Gray
    border_color = ("#E4E4E7", "#27272A")     
    
    text_main = ("#09090B", "#FAFAFA")        
    text_muted = ("#71717A", "#A1A1AA")       
    
    baby_blue = ("#38BDF8", "#38BDF8")        # The Hero Accent
    baby_hover = ("#0EA5E9", "#0EA5E9")       
    
    btn_neutral = ("#18181B", "#FAFAFA")      
    btn_neutral_hover = ("#27272A", "#E4E4E7")
    btn_neutral_text = ("#FAFAFA", "#09090B") 
    
    btn_light = ("#F4F4F5", "#27272A")        
    btn_light_hover = ("#E4E4E7", "#3F3F46")
    danger_red = ("#EF4444", "#EF4444")

    ctk.set_appearance_mode("dark") 

    app = ctk.CTk(fg_color=bg_color)
    app.title("KingKev EIS Analyzer")
    
    # ---------------------------------------------------------
    # NEW GEOMETRY: Taller to accommodate massive whitespace
    # ---------------------------------------------------------
    app.geometry("960x720")
    app.minsize(850, 650)
    
    folders = []
    list_labels = []

    # ==========================================
    # TOP BAR (Centered Theme Toggle)
    # ==========================================
    top_bar = ctk.CTkFrame(app, fg_color="transparent")
    top_bar.pack(fill="x", pady=(25, 0))

    def toggle_theme():
        if ctk.get_appearance_mode() == "Dark":
            ctk.set_appearance_mode("Light")
            theme_btn.configure(text="🌙 Dark Mode")
        else:
            ctk.set_appearance_mode("Dark")
            theme_btn.configure(text="☀️ Light Mode")

    theme_btn = ctk.CTkButton(top_bar, text="☀️ Light Mode", command=toggle_theme,
                              fg_color="transparent", hover_color=btn_light_hover, text_color=text_muted,
                              font=("Segoe UI", 12, "bold"), width=40, corner_radius=12)
    theme_btn.pack(anchor="center")

    # ==========================================
    # MASSIVE HERO HEADER (Extreme Whitespace)
    # ==========================================
    header_frame = ctk.CTkFrame(app, fg_color="transparent")
    # MASSIVE PADDING: 60px above, 60px below to isolate the text completely
    header_frame.pack(fill="x", pady=(60, 60))
    
    # Massive Size 52 font, single line, commanding the space
    ctk.CTkLabel(header_frame, text="EIS Data Analyzer", 
                 font=("Segoe UI", 52, "bold"), text_color=text_main).pack(anchor="center")
                 
    ctk.CTkLabel(header_frame, text="by Ali Islam Ali Obeidat", 
                 font=("Segoe UI", 16, "bold"), text_color=baby_blue).pack(anchor="center", pady=(6, 0))

    # ==========================================
    # COMPACT DATA TRAY
    # ==========================================
    card = ctk.CTkFrame(app, fg_color=card_color, corner_radius=16, border_width=1, border_color=border_color)
    card.pack(fill="x", padx=160, pady=(0, 30))

    # Mini Toolbar (Centered Row inside the tray)
    card_toolbar = ctk.CTkFrame(card, fg_color="transparent")
    card_toolbar.pack(fill="x", padx=25, pady=(15, 5))

    ctk.CTkLabel(card_toolbar, text="Measurement Directories", font=("Segoe UI", 14, "bold"), text_color=text_main).pack(side="left")

    def save_session():
        if not folders: return messagebox.showinfo("Empty", "No folders to save.")
        try:
            with open(SESSION_FILE, 'w') as f: json.dump(folders, f)
            messagebox.showinfo("Saved", f"Session saved successfully!\n({len(folders)} folders)")
        except Exception as e: messagebox.showerror("Error", f"Could not save session:\n{e}")

    def load_session():
        if not os.path.exists(SESSION_FILE): return messagebox.showinfo("No Session", "No saved session found.")
        try:
            with open(SESSION_FILE, 'r') as f: saved = json.load(f)
            folders.clear()
            folders.extend([p for p in saved if os.path.exists(p)]) 
            refresh_list()
        except Exception as e: messagebox.showerror("Error", f"Could not load session:\n{e}")

    def clear_folders():
        folders.clear()
        refresh_list()

    mini_btn_style = {"fg_color": btn_light, "hover_color": btn_light_hover, "font": ("Segoe UI", 11, "bold"), "height": 26, "corner_radius": 8}
    
    ctk.CTkButton(card_toolbar, text="Clear", command=clear_folders, text_color=danger_red, width=50, **mini_btn_style).pack(side="right", padx=(5, 0))
    ctk.CTkButton(card_toolbar, text="Load", command=load_session, text_color=text_main, width=50, **mini_btn_style).pack(side="right", padx=5)
    ctk.CTkButton(card_toolbar, text="Save", command=save_session, text_color=text_main, width=50, **mini_btn_style).pack(side="right", padx=5)

    # Fixed Height Scrollable Frame
    scroll_list = ctk.CTkScrollableFrame(card, fg_color="transparent", height=160, 
                                         scrollbar_button_color=btn_light, scrollbar_button_hover_color=baby_blue)
    scroll_list.pack(fill="x", expand=False, padx=15, pady=(0, 15))

    def remove_individual_folder(folder_path):
        if folder_path in folders:
            folders.remove(folder_path)
            refresh_list()

    def refresh_list():
        for lbl in list_labels: lbl.destroy()
        list_labels.clear()
        
        if not folders:
            lbl = ctk.CTkLabel(scroll_list, text="Pipeline is empty. Click '+ Add Folders' below.", 
                               font=("Segoe UI", 14), text_color=text_muted)
            lbl.pack(pady=45)
            list_labels.append(lbl)
        else:
            for f in folders:
                row = ctk.CTkFrame(scroll_list, fg_color="transparent")
                row.pack(fill="x", pady=2)
                
                icon = ctk.CTkLabel(row, text="📁", font=("Segoe UI", 16), text_color=baby_blue)
                icon.pack(side="left", padx=(5, 10))
                
                lbl = ctk.CTkLabel(row, text=os.path.basename(f), font=("Consolas", 13), text_color=text_main)
                lbl.pack(side="left")
                
                del_btn = ctk.CTkButton(row, text="✕", width=28, height=28, fg_color="transparent", 
                                        hover_color=btn_light, text_color=text_muted, font=("Segoe UI", 14),
                                        command=lambda path=f: remove_individual_folder(path))
                del_btn.pack(side="right")
                list_labels.append(row)

    refresh_list() 

    # ==========================================
    # BOTTOM CONTROLS (Centered & Pill-Shaped)
    # ==========================================
    controls_frame = ctk.CTkFrame(app, fg_color="transparent")
    controls_frame.pack(fill="x", pady=(0, 20))

    # JC Mode Toggle
    jc_mode_var = ctk.BooleanVar(value=False)
    jc_switch = ctk.CTkSwitch(controls_frame, text="Enable JC Mode (PNGs)", variable=jc_mode_var,
                              font=("Segoe UI", 13, "bold"), text_color=text_main,
                              progress_color=baby_blue, button_color="#FFFFFF", button_hover_color=btn_light_hover)
    jc_switch.pack(anchor="center", pady=(0, 30))

    # Action Buttons (Centered Row)
    action_btn_frame = ctk.CTkFrame(controls_frame, fg_color="transparent")
    action_btn_frame.pack(anchor="center")

    def add_folders():
        added_any = False
        while True:
            f = filedialog.askdirectory(title="Select a KingKev Folder (Click Cancel when done)")
            if not f: break
            if f not in folders:
                folders.append(f)
                added_any = True
        if added_any: refresh_list()

    def start_analysis():
        if not folders: messagebox.showwarning("Missing Data", "Please add at least one folder.")
        else: app.quit()

    app.protocol("WM_DELETE_WINDOW", lambda: (folders.clear(), app.quit()))

    # Sleek Pill-Shaped Buttons
    btn_add = ctk.CTkButton(action_btn_frame, text="+ Add Folders", command=add_folders, 
                            fg_color=btn_neutral, hover_color=btn_neutral_hover, text_color=btn_neutral_text, 
                            font=("Segoe UI", 15, "bold"), corner_radius=24, width=150, height=48)
    btn_add.pack(side="left", padx=10)

    btn_run = ctk.CTkButton(action_btn_frame, text="Run Pipeline", command=start_analysis, 
                            fg_color=baby_blue, hover_color=baby_hover, text_color="#09090B", 
                            font=("Segoe UI", 15, "bold"), corner_radius=24, width=160, height=48)
    btn_run.pack(side="left", padx=10)

    # --- Start GUI Loop ---
    app.mainloop()

    # --- Post-GUI Execution ---
    jc_mode = jc_mode_var.get()
    app.destroy()

    if folders:
        import plotly.express as px 
        print(f"\nProcessing {len(folders)} Folder(s):")
        for f in folders: print(f" - {f}")
            
        config = {"custom_colors": px.colors.qualitative.Plotly}
        analyzer = KingKevEISAnalyzer(folders, config)
        analyzer.parse_master_log()
        analyzer.parse_eis_files()
        analyzer.generate_kingkev_plots(jc_mode=jc_mode)
    else: 
        print("Analysis Cancelled. Exiting.")

if __name__ == '__main__':
    main()